## Setup

In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
import plotly.io as pio
pio.renderers.default = 'jupyterlab' # or 'notebook_connected'
from importlib import reload

from math import *

import testObjects

reload(testObjects)

<module 'testObjects' from '/Users/espillar/Desktop/vibevolts/testObjects.py'>

In [5]:
def amag(m):  return(10**(-0.4 * m))
def mag(f): return( -2.5 * log10(f))
print(amag(5))
print(mag(100))

0.01
-5.0


V band solar brightness is -26.74 accounding to gemini, radiometry_data has it as -26.78. Agreement. 

Gemini makes this as $5.1 \times 10^{20}$ photons per second per square meter.  Using the constants in my head I get 

In [6]:
print(amag(-26.74) * 866000 * 10000)

4.300489503759908e+20


## Code: 
calculate_stellit_magnitude(rad, dist, alb, phs_deg), 
vbandphots(mag)

Which I think is good  agreement.  Now lets get the lambertian flux at earth. Well, I can get a complete thing out of gemini- I know I'm being lazy, but this is expedient.  

In [7]:
import numpy as np

def calculate_satellite_magnitude(radius, distance, albedo, phase_angle_deg):
    """
    Calculates the apparent V-band magnitude of a spherical satellite.
    """
    m_sun = -26.74
    alpha_rad = np.radians(phase_angle_deg)
    
    # 1. Get the Lambertian Phase Function value
    # Note: We divide by the peak value at alpha=0 (2/(3*pi)*pi) 
    # to use it as a scaling factor between 0 and 1.
    phi = (1 / np.pi) * ((np.pi - alpha_rad) * np.cos(alpha_rad) + np.sin(alpha_rad))
    
    # 2. Calculate the brightness ratio
    # (Albedo * Area_ratio * Phase)
    # The (2/3) comes from the integration of a diffuse sphere's brightness
    brightness_ratio = albedo * (2/3) * (radius**2 / distance**2) * phi
    
    # 3. Convert to magnitude
    mag = m_sun - 2.5 * np.log10(brightness_ratio)
    
    return mag

# Example: A 2-meter radius satellite at 550km altitude (LEO) 
# at a 30-degree phase angle with an albedo of 0.5
sat_mag = calculate_satellite_magnitude(radius=2, distance=100000000, albedo=0.2, phase_angle_deg=0)

print(f"Satellite Apparent Magnitude: {sat_mag:.2f}")

Satellite Apparent Magnitude: 13.94


Converted this to LaTeX and compared algorithm with Cognion, it looks good.

In [8]:
def vbandphots(magnitude):
    ''' The output here will be in photons per cm2 per second'''
    p = 866000 * amag(magnitude)
    return(p) 

In [9]:
def v_mag_to_photon_flux(v_mag, band_width_angstroms=890):
    """
    Converts a V-band magnitude to photon flux (photons / cm^2 / s).
    
    Physics Constants based on Bessell (1979) / Vega system:
    - V-band Effective Wavelength: ~5500 Angstroms
    - V-band Zero Point (Flux Density at V=0): ~3.63e-9 erg / cm^2 / s / A
    - V-band FWHM (Bandwidth): ~890 Angstroms (default)
    
    Args:
        v_mag (float): The apparent V magnitude.
        band_width_angstroms (float): The filter width to integrate over. 
                                      Defaults to 890 A (standard Johnson V).
                                      
    Returns:
        float: Integrated photon flux (photons / cm^2 / s)
    """
    
    # 1. Constants
    # Zero point flux density in energy units (erg / cm^2 / s / A)
    # Source: Bessell (1979) for Vega
    F_0_energy = 3.63e-9 
    
    # Energy of a single photon at 5500 Angstroms
    # E = hc / lambda
    h = 6.626e-27 # Planck constant (erg * s)
    c = 2.998e10  # Speed of light (cm / s)
    wavelength_cm = 5500 * 1e-8
    
    E_photon = (h * c) / wavelength_cm # approx 3.61e-12 erg
    
    # 2. Convert Zero Point Energy Flux to Photon Flux Density
    # (photons / cm^2 / s / A)
    N_0_density = F_0_energy / E_photon 
    
    # 3. Calculate Flux Density for the specific magnitude
    # Formula: F = F0 * 10^(-0.4 * m)
    flux_density = N_0_density * 10**(-0.4 * v_mag)
    
    # 4. Integrate over the bandwidth to get total photons per second per cm^2
    total_photon_flux = flux_density * band_width_angstroms
    
    return float(total_photon_flux)

In [10]:
print(vbandphots(20), "  ", v_mag_to_photon_flux(20))

0.008660000000000001    0.008944915888185443


In [11]:
def photoelectorns(r, range, albedo, phaseangle, itime, aper, qe):
    electrons = vbandphots(calculate_satellite_magnitude(r, range, albedo, phaseangle) )  * ( # phots per cm2 per sec
      10 *  pi * aper**2 * qe ) # times second time pi r^2 telecope aperture times 0.2 efficiency
    return electrons

In [12]:
def backgroundshotnoise(pixsizeasec, aperrad, backmagV, itime,qe):
    """we are assuming V band for the background
    backmagV is magnitudes per square arcsecond
    pixsizeasec is one pixel side units arcseconds
    aperrad is in cm
    This is ONLY the blp"""
    pe = vbandphots(backmagV) * itime * pixsizeasec**2 * pi * aperrad**2 * qe
    return(sqrt(pe))

## Electrons received, read noise, SNR

In [13]:
r = 1 # 1m radius satellite, 
satrange = 100000000 # 100000000 m
albedo = 0.2 # 0.2 albedo
phaseangle = 0 #  0 degree phase angle
itime = 10 #Integration Time
aperrad = 50 # 50 cm aperture
qe = 0.2 # Net conversions o incident photons to photoelectrons
pixsizeasec = 1 # Pixel size in arcsec
backmagV  = 23 # Magnitudes per square arcsecond


print(calculate_satellite_magnitude(r, satrange, albedo, phaseangle), "  satellite magnitude")
signal = photoelectorns(r, satrange, albedo, phaseangle, itime, aperrad, qe)
noise = backgroundshotnoise(pixsizeasec, aperrad, backmagV, itime, qe)
print(signal, noise, signal/noise)

15.447653158479252   satellite magnitude
9006.924154568074 2.929671218546315 3074.380530330381


## Code Tests

In [6]:
fig = testObjects.demoFixed()

--- Running scandetector ---
sun, space, sky  4.529e+20, 2.360e+11, 6.499e+11
SignAl, noise, snr, integration time 

5.967e-15, 1.547e+05, 3.858e-20, 1.291e-01
5.967e-17, 1.547e+05, 3.858e-22, 1.291e-01
Output of scandetector: 0


In [19]:
print(1+1)

2
